# Inspecting downloaded ZTF data

This notebook is a tour of the three data products this repo downloads for a
target, and of the `src/` helpers used to work with them.

**Prerequisite** — download data for the default target (the Crab Nebula) first:

```bash
python scripts/download_cutouts.py --ra 83.633 --dec 22.015 --size 64
```

Per matching exposure, that stores three files under `$ZTFDATA` (default
`~/datasets/ZTF`, symlinked as `data/ztf` in this repo):

| product | contents |
|---|---|
| `sciimg.fits` | science image, cut out around the target |
| `sciimgdaopsfcent.fits` | pipeline PSF rendering at quadrant center |
| `sciimgdao.psf` | DAOPHOT text model of the PSF's spatial variation |

The query itself (which exposures cover the target) is cached as a CSV under
`$ZTFDATA/query/`, keyed by a hash of the query parameters.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
from astropy.io import fits
from matplotlib import pyplot as plt

from src.daophot import parse_daophot_psf
from src.psf import evaluate_psf_at_position
from src.ztf_data import load_query, local_paths, metatable_wcs

## 1. Load the cached query

`load_query` reuses the CSV cached by `download_cutouts.py` — same arguments,
no new IRSA query. Its metatable has one row per exposure (seeing, filter,
airmass, WCS of the full quadrant, ...).

In [ ]:
ra, dec = 83.633, 22.015   # Crab Nebula (see data/targets.csv)
size = 64                  # cutout half-size (arcsec)
max_seeing = 1.7

zquery = load_query(ra, dec, size, max_seeing)
paths = local_paths(zquery)

print(f'{len(zquery.metatable)} exposures,',
      {k: len(v) for k, v in paths.items()}, 'files on disk')
zquery.metatable[['obsdate', 'filtercode', 'seeing', 'airmass', 'maglimit']].head()

## 2. Science cutouts

A random sample of cutouts, asinh-stretched. Occasional cutouts come out
smaller than requested when the target sits near a quadrant edge — filter on
shape before stacking anything.

In [ ]:
def normalize(x):
    lo, hi = np.quantile(x, [0.5, 0.99])
    return (x - lo) / (hi - lo)

cutouts = [d for f in paths['sciimg.fits']
           if (d := fits.getdata(f)).shape == (size, size)]
print(f'{len(cutouts)} / {len(paths["sciimg.fits"])} cutouts have the full '
      f'{size}x{size} shape')

grid = 3
rng = np.random.default_rng(0)
sample = rng.choice(len(cutouts), size=grid * grid, replace=False)
mosaic = np.block([[normalize(cutouts[j]) for j in sample[i*grid:(i+1)*grid]]
                   for i in range(grid)])

plt.figure(figsize=(8, 8))
plt.imshow(np.asinh(mosaic), cmap='gray', origin='lower')
plt.axis('off')
plt.title(f'{grid}x{grid} random cutouts around ({ra}, {dec})')
plt.show()

## 3. Center PSF renderings

`sciimgdaopsfcent.fits` is the pipeline's own PSF rendering at the quadrant
center. It preserves the real asymmetries of the optics, which is why the
repo uses it as the base PSF rather than re-rendering the analytic DAOPHOT
model.

In [ ]:
sample = rng.choice(len(paths['sciimgdaopsfcent.fits']), size=4, replace=False)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, j in zip(axes, sample):
    psf = fits.getdata(paths['sciimgdaopsfcent.fits'][j])
    seeing = zquery.metatable.iloc[int(j)]['seeing']
    ax.imshow(psf, origin='lower')
    ax.set_title(f'seeing={seeing:.2f}"')
    ax.axis('off')
plt.suptitle('Pipeline center-PSF renderings')
plt.show()

## 4. Position-matched PSF

The center rendering is only exact at the quadrant center. `sciimgdao.psf`
(a DAOPHOT-II text file, parsed by `src/daophot.py`) provides lookup-table
planes describing how the PSF varies across the quadrant.
`src/psf.py:evaluate_psf_at_position` adds that spatial correction to the
center rendering at the target's quadrant position.

This is exactly what `scripts/build_psf_pairs.py` does for every exposure;
below we do it by hand for one exposure to see the size of the correction.

In [ ]:
i = 0
row = zquery.metatable.iloc[i]
x_quad, y_quad = (float(v) for v in metatable_wcs(row).all_world2pix(ra, dec, 0))

psf_cen = fits.getdata(paths['sciimgdaopsfcent.fits'][i]).astype(float)
psf_cen /= psf_cen.sum()
psf_info = parse_daophot_psf(paths['sciimgdao.psf'][i])
psf = evaluate_psf_at_position(psf_cen, psf_info, x_quad, y_quad)

correction = psf - psf_cen
corr_max = np.abs(correction).max()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title, kw in [
    (axes[0], psf_cen, 'center PSF', {}),
    (axes[1], psf, f'matched PSF at ({x_quad:.0f},{y_quad:.0f})', {}),
    (axes[2], correction, f'correction (max={corr_max:.4f})',
     dict(cmap='RdBu_r', vmin=-corr_max, vmax=corr_max)),
]:
    im = ax.imshow(img, origin='lower', **kw)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()